<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/w05_model_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Logistic Regression as the readable first learned model, followed by Random Forest as a stronger nonlinear comparison.

**Why it fits this lane:** the lane asks which content should be reviewed first for possible refresh. We have a yes/no future decline label, so a classifier can produce a probability used as a ranking score. Logistic Regression is a simple, inspectable benchmark; Random Forest is added only to test whether nonlinear interactions earn a meaningful improvement.

The Week-4 baseline remains the fixed, transparent rule: stale + position slipping + visible impressions. The learned models must beat or lose to that rule on the **same March test rows and the same future-decline label**.

In [10]:

# Setup: connect to the gated FlyRank warehouse.
# The token is read from the Colab Secret HF_TOKEN; it is never stored in notebook code.
import os
import json
import numpy as np
import pandas as pd
import duckdb

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "HF_TOKEN is not available. In Colab, add your Hugging Face READ token "
        "as a Secret named HF_TOKEN and enable notebook access, then run again."
    ) from e

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Week-4 baseline used March 2026 as its development snapshot.
# We keep March as the held-out test snapshot and use February as training.
# The April outcome is never used as a feature.
MONTHS = {
    "2026-01": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-01/*.parquet')",
    "2026-02": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')",
    "2026-03": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    "2026-04": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')",
}

# Verify the partitions before building features.
for month, src in MONTHS.items():
    chk = con.sql(f"""
        SELECT COUNT(*) AS n, MIN(report_date) AS min_date, MAX(report_date) AS max_date
        FROM {src}
    """).fetchone()
    print(month, chk)

SEED = 42
TEST_SIZE = 0.25
DECLINE_THRESHOLD = 0.80
MIN_IMPRESSIONS = 100
MIN_DAYS = 20


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-01 (7890817, datetime.date(2026, 1, 1), datetime.date(2026, 1, 31))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-02 (7355108, datetime.date(2026, 2, 1), datetime.date(2026, 2, 28))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-03 (9841378, datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-04 (10424730, datetime.date(2026, 4, 1), datetime.date(2026, 4, 30))


## 2. Split design

I use a **time-aware development design**: February 2026 is the training snapshot and March 2026 is the held-out test snapshot. The March snapshot matches the Week-4 baseline's development month; April is used only to create the test label.

Within the training snapshot, I also use a **client-group validation split** to avoid putting pages from the same client into both training and validation. The final baseline-vs-model comparison is on the untouched March test snapshot, so every method sees the same rows and the same future-decline label.

This is stricter than a random row split and better matches the question: rank content for future review while avoiding client memorization.

In [11]:

# Build two monthly snapshots:
# - February snapshot -> March future decline = training examples
# - March snapshot -> April future decline = held-out test examples
#
# Features use only the snapshot month and its immediately preceding 30 days.
# The next month is used ONLY to create y, never as an input feature.

daily_parts = []
for month, src in MONTHS.items():
    daily_parts.append(f"""
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_data_available,
            gsc_impressions,
            gsc_clicks,
            CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END AS gsc_avg_position
        FROM {src}
        WHERE gsc_data_available IS TRUE
    """)

daily_union = " UNION ALL ".join(daily_parts)

monthly = con.sql(f"""
WITH daily AS (
    {daily_union}
)
SELECT
    DATE_TRUNC('month', report_date) AS month,
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    COUNT(DISTINCT report_date) AS gsc_days
FROM daily
GROUP BY 1,2,3
""").df()

monthly["month"] = pd.to_datetime(monthly["month"]).dt.strftime("%Y-%m")
print("Monthly aggregate rows:", f"{len(monthly):,}")

content = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    content_updated_date,
    search_volume,
    word_count,
    content_type,
    is_published,
    is_deleted
FROM {CONTENT}
""").df()

def make_snapshot(snapshot_month):
    # Previous and current 30-day windows are represented by adjacent monthly partitions.
    prev_month = (pd.Period(snapshot_month, freq="M") - 1).strftime("%Y-%m")
    next_month = (pd.Period(snapshot_month, freq="M") + 1).strftime("%Y-%m")

    cur = monthly[monthly["month"] == snapshot_month].copy()
    prev = monthly[monthly["month"] == prev_month].copy()
    nxt = monthly[monthly["month"] == next_month].copy()

    cur = cur.rename(columns={
        "impressions":"impressions_current30",
        "clicks":"clicks_current30",
        "avg_position":"position_current30",
        "gsc_days":"gsc_days_current30"
    })
    prev = prev.rename(columns={
        "impressions":"impressions_prev30",
        "clicks":"clicks_prev30",
        "avg_position":"position_prev30",
        "gsc_days":"gsc_days_prev30"
    })
    nxt = nxt.rename(columns={
        "impressions":"impressions_next30",
        "clicks":"clicks_next30",
        "avg_position":"position_next30",
        "gsc_days":"gsc_days_next30"
    })

    x = cur.merge(prev, on=["client_hash_id","content_hash_id"], how="inner")
    x = x.merge(nxt[[
        "client_hash_id","content_hash_id","impressions_next30","clicks_next30",
        "position_next30","gsc_days_next30"
    ]], on=["client_hash_id","content_hash_id"], how="inner")
    x = x.merge(content, on=["client_hash_id","content_hash_id"], how="left")

    end_date = pd.Timestamp(snapshot_month + "-01") + pd.offsets.MonthEnd(0)
    x["staleness_days"] = (
        end_date - pd.to_datetime(x["content_updated_date"])
    ).dt.days
    x["position_slip"] = x["position_current30"] - x["position_prev30"]
    x["ctr_current30"] = x["clicks_current30"] / x["impressions_current30"].replace(0, np.nan)
    x["ctr_prev30"] = x["clicks_prev30"] / x["impressions_prev30"].replace(0, np.nan)

    # Decision-time eligibility and label-quality requirements.
    x = x[
        (x["is_published"] == True) &
        (x["is_deleted"] == False) &
        (x["gsc_days_prev30"] >= MIN_DAYS) &
        (x["gsc_days_current30"] >= MIN_DAYS) &
        (x["gsc_days_next30"] >= MIN_DAYS) &
        (x["impressions_prev30"] >= MIN_IMPRESSIONS) &
        (x["impressions_current30"] >= MIN_IMPRESSIONS)
    ].copy()

    # Future outcome: next 30d impressions are <80% of current 30d impressions.
    x["is_declining"] = (
        x["impressions_next30"] < DECLINE_THRESHOLD * x["impressions_current30"]
    ).astype(int)
    x["snapshot_month"] = snapshot_month
    return x

train_df = make_snapshot("2026-02")
test_df = make_snapshot("2026-03")

print("Training snapshot:", f"{len(train_df):,}", "rows")
print("Test snapshot:", f"{len(test_df):,}", "rows")
print("Training decline rate:", f"{train_df['is_declining'].mean():.3f}")
print("Test decline rate:", f"{test_df['is_declining'].mean():.3f}")

# Sanity checks: no future-window fields are included in model features below.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly aggregate rows: 646,601


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training snapshot: 53,140 rows
Test snapshot: 61,032 rows
Training decline rate: 0.178
Test decline rate: 0.540


## 3. Train + compare vs my baseline

The learned models are trained on February's pre-decision features and a March future-decline label. The final comparison uses March features and the April future-decline label.

The Week-4 baseline is recreated **inside this notebook** on the exact same March test rows. The primary ranking metrics are ROC AUC, average precision, Precision@20, and Precision@50, with the test base rate shown alongside them.

In [18]:

# Model features: all are knowable at the March decision point.

feature_cols = [
    "impressions_prev30",
    "impressions_current30",
    "clicks_prev30",
    "clicks_current30",
    "position_prev30",
    "position_current30",
    "position_slip",
    "ctr_prev30",
    "ctr_current30",
    "staleness_days",
    "search_volume",
    "word_count",
]

for c in feature_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

# Remove impossible/infinite values without turning missingness into zero.
train_df[feature_cols] = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
test_df[feature_cols] = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

# Grouped validation inside training data.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED
)

tr_idx, va_idx = next(
    gss.split(
        train_df,
        train_df["is_declining"],
        groups=train_df["client_hash_id"]
    )
)

tr = train_df.iloc[tr_idx].copy()
va = train_df.iloc[va_idx].copy()

print(
    "Grouped validation clients:",
    tr["client_hash_id"].nunique(),
    "train /",
    va["client_hash_id"].nunique(),
    "validation"
)
print("Validation rows:", f"{len(va):,}")

# Fit the readable model first, then the stronger nonlinear model.
logit = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED
        )
    ),
])

rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=20,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1
        )
    ),
])

logit.fit(tr[feature_cols], tr["is_declining"])
rf.fit(tr[feature_cols], tr["is_declining"])

# Select the model using grouped validation average precision.
va_logit = logit.predict_proba(va[feature_cols])[:, 1]
va_rf = rf.predict_proba(va[feature_cols])[:, 1]

val_ap = {
    "Logistic Regression": average_precision_score(
        va["is_declining"],
        va_logit
    ),
    "Random Forest": average_precision_score(
        va["is_declining"],
        va_rf
    ),
}

print("Grouped validation average precision:")
for k, v in val_ap.items():
    print(f"  {k}: {v:.3f}")

best_name = max(val_ap, key=val_ap.get)
best_model = logit if best_name == "Logistic Regression" else rf

print("Selected model:", best_name)

# IMPORTANT:
# Final comparison is on the same March test rows as the Week-4 baseline.

y_test = test_df["is_declining"].to_numpy()

model_prob = best_model.predict_proba(
    test_df[feature_cols]
)[:, 1]

# Recreate the exact Week-4 transparent baseline rule
# on these same March test rows.

stale = (
    test_df["staleness_days"] >= 180
).astype(int)

slipping = (
    test_df["position_slip"] > 2
).astype(int)

visible = (
    test_df["impressions_current30"] >= MIN_IMPRESSIONS
).astype(int)

baseline_score = (
    stale
    * slipping
    * visible
    * test_df["impressions_current30"]
)

# Both baseline_score and model_prob are already aligned
# to the exact same test_df rows.

baseline_order = np.argsort(
    -baseline_score.to_numpy(),
    kind="stable"
)

model_order = np.argsort(
    -model_prob,
    kind="stable"
)

y_eval = y_test

def precision_at(order, k):
    k = min(k, len(order))
    return float(y_eval[order[:k]].mean()) if k else np.nan


def ranking_metrics(name, scores, order):
    return {
        "method": name,
        "base_rate": float(y_eval.mean()),
        "ROC_AUC": float(roc_auc_score(y_eval, scores)),
        "Average_Precision": float(
            average_precision_score(y_eval, scores)
        ),
        "Precision@20": precision_at(order, 20),
        "Precision@50": precision_at(order, 50),
    }


comparison = pd.DataFrame([
    ranking_metrics(
        "Week-4 baseline",
        baseline_score.to_numpy(),
        baseline_order
    ),
    ranking_metrics(
        best_name,
        model_prob,
        model_order
    ),
])

print("\nMODEL VS BASELINE — SAME MARCH TEST ROWS + SAME LABEL")
print(comparison.to_string(index=False))

# Error analysis on the selected model.

test_eval = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "is_declining",
        "impressions_prev30",
        "impressions_current30",
        "position_prev30",
        "position_current30",
        "position_slip",
        "staleness_days",
        "search_volume",
    ]
].copy()

test_eval["model_probability"] = model_prob

test_eval["predicted"] = (
    model_prob >= 0.5
).astype(int)

test_eval["error_type"] = np.select(
    [
        (test_eval["predicted"] == 1)
        & (test_eval["is_declining"] == 0),

        (test_eval["predicted"] == 0)
        & (test_eval["is_declining"] == 1),
    ],
    [
        "false_positive",
        "false_negative",
    ],
    default="correct"
)

print("\nError counts:")
print(
    test_eval["error_type"]
    .value_counts()
    .to_string()
)

print("\nThree concrete false positives:")
print(
    test_eval[
        test_eval["error_type"] == "false_positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(3)
    .to_string(index=False)
)

print("\nThree concrete false negatives:")
print(
    test_eval[
        test_eval["error_type"] == "false_negative"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .head(3)
    .to_string(index=False)
)

# Permutation importance on grouped validation,
# using average precision.

perm = permutation_importance(
    best_model,
    va[feature_cols],
    va["is_declining"],
    scoring="average_precision",
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

print("\nTop permutation-importance features:")
print(
    importance.head(10).to_string(index=False)
)

# Save reproducible receipts.

out_dir = os.path.join(
    os.getcwd(),
    "work",
    "outputs"
)

os.makedirs(
    out_dir,
    exist_ok=True
)

comparison.to_csv(
    os.path.join(
        out_dir,
        "w05_model_vs_baseline.csv"
    ),
    index=False
)

importance.to_csv(
    os.path.join(
        out_dir,
        "w05_permutation_importance.csv"
    ),
    index=False
)

metrics_receipt = {
    "seed": SEED,
    "train_snapshot": "2026-02 -> March label",
    "test_snapshot": "2026-03 -> April label",
    "label": "next30 impressions < 0.80 * current30 impressions",
    "min_impressions": MIN_IMPRESSIONS,
    "min_gsc_days": MIN_DAYS,
    "selected_model": best_name,
    "validation": (
        "GroupShuffleSplit by client_hash_id "
        "on training snapshot"
    ),
    "comparison": comparison.to_dict(
        orient="records"
    ),
}

with open(
    os.path.join(
        out_dir,
        "w05_metrics.json"
    ),
    "w"
) as f:
    json.dump(
        metrics_receipt,
        f,
        indent=2
    )

print(
    "\nReceipts written to:",
    out_dir
)


Grouped validation clients: 18 train / 6 validation
Validation rows: 27,575
Grouped validation average precision:
  Logistic Regression: 0.227
  Random Forest: 0.216
Selected model: Logistic Regression

MODEL VS BASELINE — SAME MARCH TEST ROWS + SAME LABEL
             method  base_rate  ROC_AUC  Average_Precision  Precision@20  Precision@50
    Week-4 baseline   0.540454 0.500000           0.540454          0.95          0.98
Logistic Regression   0.540454 0.517839           0.554111          0.55          0.54

Error counts:
error_type
correct           31639
false_negative    15208
false_positive    14185

Three concrete false positives:
         client_hash_id          content_hash_id  is_declining  impressions_prev30  impressions_current30  position_prev30  position_current30  position_slip  staleness_days  search_volume  model_probability  predicted     error_type
client_73cda7b4e4f265ea content_512dbad65bd5ade9             0            167303.0               154358.0         2.9

## 4. Errors and interpretation

The final code reports:

- false positives and false negatives, including three concrete examples of each;
- permutation importance from the grouped validation split;
- the top features and whether their relationship is plausible;
- the model-vs-baseline table on the same held-out March snapshot.

A useful result is not automatically the most complex model. If Random Forest does not materially improve the ranking metrics over Logistic Regression, the simpler model is the better teaching result. Likewise, if the learned model does not beat the Week-4 rule, that is a valid finding rather than a reason to tune until it does.

In [19]:

# Explicit self-checks and leakage audit.
future_or_label_names = {
    "is_declining", "impressions_next30", "clicks_next30",
    "position_next30", "gsc_days_next30", "trend_pct",
    "trend_direction", "is_declining_label"
}
assert future_or_label_names.isdisjoint(set(feature_cols)), "Leakage: future/label-derived feature detected."

# Product decision outputs are not model features.
product_flags = {"health_score", "priority_score", "action_type", "decision_flag"}
assert product_flags.isdisjoint(set(feature_cols)), "Leakage: product decision output detected."

# The final test comparison is exactly the March snapshot and its April outcome.
assert set(test_df["snapshot_month"]) == {"2026-03"}
assert len(comparison) == 2
assert all(c in comparison.columns for c in ["Precision@20","Precision@50","Average_Precision"])

print("SELF-CHECK")
print("- Model features are pre-decision March/February fields only: PASS")
print("- Future April fields are label-only: PASS")
print("- Week-4 baseline and model use the same March test rows: PASS")
print("- Client IDs are grouping only, never features: PASS")
print("- Final June sample used: NO")
print("- Random seed fixed:", SEED)
print("- Output receipts written under work/outputs/: PASS")


SELF-CHECK
- Model features are pre-decision March/February fields only: PASS
- Future April fields are label-only: PASS
- Week-4 baseline and model use the same March test rows: PASS
- Client IDs are grouping only, never features: PASS
- Final June sample used: NO
- Random seed fixed: 42
- Output receipts written under work/outputs/: PASS


## Self-check

- [x] Every section is filled — method, split, model comparison, and error interpretation.
- [x] The code compares the Week-4 baseline and learned model on the same held-out March rows and future label.
- [x] The split is time-aware, with grouped client validation inside training.
- [x] No future-window or label-derived columns are model features.
- [x] Client IDs are used only for grouping.
- [x] The notebook writes reproducibility receipts under `work/outputs/`.
- [ ] **Run Runtime → Run all in Colab before submission** and confirm all cells execute without errors.
- [ ] Commit the executed notebook to `work/notebooks/w05_model.ipynb` in your own repository.


In [ ]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("Token found:", token.startswith("hf_"))

In [14]:
print("Variables containing test/validation:")
print([x for x in globals() if any(w in x.lower() for w in ["test", "valid", "eval"])])

Variables containing test/validation:
['TEST_SIZE', 'test_df', 'y_test']


In [15]:
print("Relevant variables:")
for name in ["X_tr", "X_val", "X_train", "X_valid", "y_tr", "y_val", "y_test",
             "baseline_score", "model_prob", "model_order"]:
    if name in globals():
        try:
            print(name, type(globals()[name]), len(globals()[name]))
        except:
            print(name, type(globals()[name]))

Relevant variables:
y_test <class 'numpy.ndarray'> 61032
baseline_score <class 'pandas.core.series.Series'> 61032
model_prob <class 'numpy.ndarray'> 61032
model_order <class 'numpy.ndarray'> 61032
